# 02 · Classical DIP Walkthrough — Kerala 2018

Objective: walk through every classical DIP technique used in the project on the pre/post Kerala composites, with captioned figures suitable for the final report.

Sections:
1. Load pre/post rasters
2. Spectral indices (NDWI, MNDWI, NDVI, AWEInsh, AWEIsh)
3. Thresholding (Otsu, Triangle, Yen, Li, adaptive)
4. Morphological post-processing
5. Change detection (differencing, ratioing, PCA, CVA)
6. Frequency-domain analysis (FFT sanity check)
7. Full classical baseline run + saved mask

Every figure is saved to `reports/figures/` at 150 DPI.

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

REPO_ROOT = Path.cwd() if (Path.cwd() / 'PRD.md').exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from src.dip import indices, thresholding, morphology, change_detection, filters
from src.data.aoi import load_aoi

aoi = load_aoi('configs/kerala_2018.yaml')
FIG_DIR = Path('reports/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.tight_layout()
    out = FIG_DIR / name
    fig.savefig(out, dpi=150, bbox_inches='tight')
    print(f'saved {out}')

def stretch(x, lo=2, hi=98):
    out = np.zeros_like(x, dtype=np.float32)
    for i in range(x.shape[0]):
        a, b = np.percentile(x[i], [lo, hi])
        out[i] = np.clip((x[i] - a) / (b - a + 1e-9), 0, 1)
    return out

def rgb(arr, ids=(2, 1, 0)):  # default true-colour from DDA band order
    return np.transpose(stretch(arr[list(ids)]), (1, 2, 0))

## 1 · Load rasters

In [ ]:
PRE = Path('data/raw/kerala_2018/kerala_2018_pre.tif')
POST = Path('data/raw/kerala_2018/kerala_2018_post.tif')

for p in (PRE, POST):
    assert p.exists(), f'Missing {p}. Run src.data.gee_download first.'

with rasterio.open(PRE) as src:
    pre = src.read().astype(np.float32)
with rasterio.open(POST) as src:
    post = src.read().astype(np.float32)

print('pre', pre.shape, 'post', post.shape)

## 2 · Spectral indices

Display all five indices side-by-side for the post-event scene. Compare against MNDWI_pre to see where the flood emerged.

In [ ]:
post_idx = indices.compute_all(post)
pre_mndwi = indices.mndwi(pre)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes[0, 0].imshow(rgb(post)); axes[0, 0].set_title('Post · RGB'); axes[0, 0].axis('off')
for ax, (name, arr) in zip(axes.ravel()[1:], post_idx.items()):
    im = ax.imshow(arr, cmap='RdYlBu', vmin=-1, vmax=1)
    ax.set_title(f'{name.upper()} · post'); ax.axis('off')
plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.65, label='index value')
save(fig, 'phase2_indices_post.png'); plt.show()

## 3 · Thresholding on MNDWI

Compare every global-histogram method plus the adaptive method on the same MNDWI_post field.

In [ ]:
m = post_idx['mndwi']
results = {
    'Otsu':      thresholding.otsu(m),
    'Triangle':  thresholding.triangle(m),
    'Yen':       thresholding.yen(m),
    'Li':        thresholding.li(m),
    'Adaptive':  thresholding.adaptive(m, block_size=51),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0, 0].hist(m[np.isfinite(m)].ravel(), bins=80, color='steelblue')
for name, res in results.items():
    axes[0, 0].axvline(res.value, linestyle='--', label=f'{name}={res.value:.2f}')
axes[0, 0].set_title('MNDWI histogram + chosen thresholds'); axes[0, 0].legend(fontsize=7)

for ax, (name, res) in zip(axes.ravel()[1:], results.items()):
    ax.imshow(res.mask, cmap='Blues'); ax.set_title(f'{name} · water%={res.mask.mean()*100:.1f}'); ax.axis('off')
save(fig, 'phase2_thresholding.png'); plt.show()

## 4 · Morphological post-processing

Start from the Otsu mask and apply opening → closing → small-object/hole removal. Each stage shown.

In [ ]:
raw = results['Otsu'].mask
cleaned = morphology.clean(raw)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw, cmap='Blues'); axes[0].set_title(f'raw · {raw.sum()} px'); axes[0].axis('off')
axes[1].imshow(cleaned, cmap='Blues'); axes[1].set_title(f'after clean() · {cleaned.sum()} px'); axes[1].axis('off')
axes[2].imshow(rgb(post)); axes[2].imshow(morphology.boundary(cleaned), cmap='Reds', alpha=0.6)
axes[2].set_title('Boundary overlay on RGB'); axes[2].axis('off')
save(fig, 'phase2_morphology.png'); plt.show()

## 5 · Change detection

Four classical techniques on the same pre/post pair. All should highlight NEW water relative to the dry baseline.

In [ ]:
d_mndwi = change_detection.mndwi_difference(pre, post)
d_diff  = change_detection.image_difference(pre, post).mean(axis=0)
d_ratio = change_detection.image_ratio(pre, post).mean(axis=0)
d_cva   = change_detection.change_vector_analysis(pre, post)
d_pca   = change_detection.pca_change(pre, post, n_components=3)

panels = {
    'ΔMNDWI': d_mndwi,
    'Image differencing (mean)': d_diff,
    'Image ratioing (mean)': d_ratio,
    'CVA magnitude': d_cva,
    'PCA last component': d_pca,
}
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes[0, 0].imshow(rgb(post)); axes[0, 0].set_title('Post · RGB'); axes[0, 0].axis('off')
for ax, (name, arr) in zip(axes.ravel()[1:], panels.items()):
    im = ax.imshow(arr, cmap='viridis')
    ax.set_title(name); ax.axis('off')
save(fig, 'phase2_change_detection.png'); plt.show()

## 6 · Frequency-domain analysis (FFT)

Sanity check: visualise the FFT magnitude spectrum of MNDWI_post. If low-frequency components dominate (as expected for natural imagery), a **spatial-domain** classical pipeline is appropriate; if there's strong periodic banding (striping), a frequency-domain notch filter would be necessary first.

In [ ]:
mm = post_idx['mndwi']
F = np.fft.fftshift(np.fft.fft2(mm - mm.mean()))
mag = np.log1p(np.abs(F))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(mm, cmap='RdYlBu', vmin=-1, vmax=1); axes[0].set_title('MNDWI_post (spatial)')
axes[1].imshow(mag, cmap='magma'); axes[1].set_title('log(|FFT|) magnitude spectrum')
for ax in axes: ax.axis('off')
save(fig, 'phase2_fft.png'); plt.show()

## 7 · Full classical baseline

Run the end-to-end pipeline and save the output mask to disk. This is the artefact Phase 3 evaluates.

In [ ]:
from src.pipelines.classical_baseline import run_classical_baseline

out = Path('data/processed/kerala_2018/flood_mask_classical.tif')
result = run_classical_baseline(PRE, POST, out)
print(result)

with rasterio.open(out) as src:
    mask = src.read(1).astype(bool)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(rgb(post)); ax[0].set_title('Post · RGB'); ax[0].axis('off')
ax[1].imshow(rgb(post)); ax[1].imshow(mask, cmap='Blues', alpha=0.55)
ax[1].set_title(f'Classical flood mask · {result.flood_area_km2:.1f} km² ({result.flood_fraction*100:.1f}%)'); ax[1].axis('off')
save(fig, 'phase2_classical_baseline.png'); plt.show()

## Phase 2 exit criteria

- [ ] `data/processed/kerala_2018/flood_mask_classical.tif` produced.
- [ ] `pytest tests/test_dip.py tests/test_preprocess.py tests/test_pipelines.py` green.
- [ ] All five figures in `reports/figures/phase2_*.png` rendered.
- [ ] Qualitative check: flooded regions visually align with low-lying districts (Alappuzha / Ernakulam / Thrissur).

Next → **Phase 3 · Baseline evaluation & ablation study**.